я надумав опрацювати краще дані по вінрейту кожної з команд - знайти standart deviation між перемогами вдома і гостях, 1. outliers - команди які краще грають вдома краще, ніж інші. тобто в який відсоток переміг замітно більший за sd. 2. також список команд, які грають краще на виїзді 3. відсортувати по сезонам і по лігам цю інформацію

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import sqlite3
import helpers as hp
conn = sqlite3.connect('database.sqlite')

In [2]:
#create df with data needed
matches_df = pd.read_sql("SELECT match_api_id, league_id, season, home_team_api_id, away_team_api_id, home_team_goal, away_team_goal FROM match", conn)  

In [3]:
#add column with home team name
matches_df = hp.team_names(matches_df, "home_team_api_id", "home_team")

In [4]:
#add column with away team name
matches_df = hp.team_names(matches_df,"away_team_api_id", "away_team")

In [5]:
#add column with result
matches_df = hp.result(matches_df)

In [6]:
matches_df.head()

,match_api_id,league_id,season,home_team_api_id,away_team_api_id,home_team_goal,away_team_goal,home_team,away_team,result
0,492473,1,2008/2009,9987,9993,1,1,KRC Genk,Beerschot AC,draw
1,492474,1,2008/2009,10000,9994,0,0,SV Zulte-Waregem,Sporting Lokeren,draw
2,492475,1,2008/2009,9984,8635,0,3,KSV Cercle Brugge,RSC Anderlecht,away_win
3,492476,1,2008/2009,9991,9998,5,0,KAA Gent,RAEC Mons,home_win
4,492477,1,2008/2009,7947,9985,1,3,FCV Dender EH,Standard de Liège,away_win


In [7]:
team_match_home = matches_df[[
    'home_team_api_id',
    'home_team',
    'league_id',
    'season', 
    'home_team_goal',
    'away_team_goal',
    'result'
    ]].rename(
    columns = {
        'home_team_api_id':"team_id",
        'home_team': "Club",
        'league_id': "League",
        'season': "Season",
        'home_team_goal': "GF",
        'away_team_goal': "GA"}
    )

In [8]:
team_match_home.head()

,team_id,Club,League,Season,GF,GA,result
0,9987,KRC Genk,1,2008/2009,1,1,draw
1,10000,SV Zulte-Waregem,1,2008/2009,0,0,draw
2,9984,KSV Cercle Brugge,1,2008/2009,0,3,away_win
3,9991,KAA Gent,1,2008/2009,5,0,home_win
4,7947,FCV Dender EH,1,2008/2009,1,3,away_win


In [9]:
#creating dataframe for further analysis
team_match_away = matches_df[[
    'away_team_api_id',
    'away_team',
    'league_id',
    'season', 
    'away_team_goal',
    'home_team_goal',
    'result'
    ]].rename(
    columns = {
        'away_team_api_id':"team_id",
        'away_team': "Club",
        'league_id': "League",
        'season': "Season",
        'away_team_goal': "GF",
        'home_team_goal': "GA"}
    )

In [10]:
team_match_away.head()

,team_id,Club,League,Season,GF,GA,result
0,9993,Beerschot AC,1,2008/2009,1,1,draw
1,9994,Sporting Lokeren,1,2008/2009,0,0,draw
2,8635,RSC Anderlecht,1,2008/2009,3,0,away_win
3,9998,RAEC Mons,1,2008/2009,0,5,home_win
4,9985,Standard de Liège,1,2008/2009,3,1,away_win


In [11]:
#changing the result column data to actual
team_match_home["result"] = team_match_home["result"].replace({
    "home_win":"W",
    "away_win":"L",
    "draw":"D"})

In [12]:
team_match_home.head()

,team_id,Club,League,Season,GF,GA,result
0,9987,KRC Genk,1,2008/2009,1,1,D
1,10000,SV Zulte-Waregem,1,2008/2009,0,0,D
2,9984,KSV Cercle Brugge,1,2008/2009,0,3,L
3,9991,KAA Gent,1,2008/2009,5,0,W
4,7947,FCV Dender EH,1,2008/2009,1,3,L


In [13]:
team_match_away["result"] = team_match_away["result"].replace({
    "home_win":"L",
    "away_win":"W",
    "draw":"D"})

In [14]:
team_match_home.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25979 entries, 0 to 25978
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   team_id  25979 non-null  int64 
 1   Club     25979 non-null  object
 2   League   25979 non-null  int64 
 3   Season   25979 non-null  object
 4   GF       25979 non-null  int64 
 5   GA       25979 non-null  int64 
 6   result   25979 non-null  object
dtypes: int64(4), object(3)
memory usage: 1.4+ MB


In [15]:
match_results = pd.concat([team_match_away,team_match_home], ignore_index = True)

In [16]:
match_results.head()

,team_id,Club,League,Season,GF,GA,result
0,9993,Beerschot AC,1,2008/2009,1,1,D
1,9994,Sporting Lokeren,1,2008/2009,0,0,D
2,8635,RSC Anderlecht,1,2008/2009,3,0,W
3,9998,RAEC Mons,1,2008/2009,0,5,L
4,9985,Standard de Liège,1,2008/2009,3,1,W


In [17]:
match_results.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51958 entries, 0 to 51957
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   team_id  51958 non-null  int64 
 1   Club     51958 non-null  object
 2   League   51958 non-null  int64 
 3   Season   51958 non-null  object
 4   GF       51958 non-null  int64 
 5   GA       51958 non-null  int64 
 6   result   51958 non-null  object
dtypes: int64(4), object(3)
memory usage: 2.8+ MB


In [18]:
team_stats = match_results.groupby(["Club","League","Season"])["result"].value_counts().unstack(fill_value=0).fillna(0).astype(int)

In [19]:
team_stats = team_stats.reset_index()

In [20]:
team_stats

result,Club,League,Season,D,L,W
0,1. FC Kaiserslautern,7809,2010/2011,7,14,13
1,1. FC Kaiserslautern,7809,2011/2012,11,19,4
2,1. FC Köln,7809,2008/2009,6,17,11
3,1. FC Köln,7809,2009/2010,11,14,9
4,1. FC Köln,7809,2010/2011,5,16,13
...,...,...,...,...,...,...
1473,Śląsk Wrocław,15722,2011/2012,5,8,17
1474,Śląsk Wrocław,15722,2012/2013,8,9,13
1475,Śląsk Wrocław,15722,2013/2014,13,10,7
1476,Śląsk Wrocław,15722,2014/2015,10,8,12


In [21]:
home_goals_for = team_match_home.groupby(["Club","League","Season"])["GF"].sum().reset_index()
home_goals_for = home_goals_for.rename(columns = {
    "GF":"GF_home"})

In [22]:
home_goals_against = team_match_home.groupby(["Club","League","Season"])["GA"].sum().reset_index()
home_goals_against = home_goals_against.rename(columns = {
    "GA":"GA_home"})

In [23]:
away_goals_for = team_match_away.groupby(["Club","League","Season"])["GF"].sum().reset_index()
away_goals_for = away_goals_for.rename(columns = {
    "GF":"GF_away"})

In [24]:
away_goals_against = team_match_away.groupby(["Club","League","Season"])["GA"].sum().reset_index()
away_goals_against = away_goals_against.rename(columns = {
    "GA":"GA_away"})

In [25]:
team_stats = team_stats.merge(
    home_goals_for, 
    on = ["Club","League","Season"], 
    how = "left")

In [26]:
team_stats = team_stats.merge(
    home_goals_against, 
    on = ["Club","League","Season"], 
    how = "left")

In [27]:
team_stats = team_stats.merge(
    away_goals_for, 
    on = ["Club","League","Season"], 
    how = "left")

In [28]:
team_stats = team_stats.merge(
    away_goals_against, 
    on = ["Club","League","Season"], 
    how = "left")

In [29]:
team_stats["GF"] = team_stats['GF_home']+team_stats['GF_away']

In [30]:
team_stats["GA"] = team_stats['GA_home']+team_stats['GA_away']

In [31]:
team_stats[team_stats["Club"] == "Manchester United"]

,Club,League,Season,D,L,W,GF_home,GA_home,GF_away,GA_away,GF,GA
800,Manchester United,1729,2008/2009,6,4,28,43,13,25,11,68,24
801,Manchester United,1729,2009/2010,4,7,27,52,12,34,16,86,28
802,Manchester United,1729,2010/2011,11,4,23,49,12,29,25,78,37
803,Manchester United,1729,2011/2012,5,5,28,52,19,37,14,89,33
804,Manchester United,1729,2012/2013,5,5,28,45,19,41,24,86,43
805,Manchester United,1729,2013/2014,7,12,19,29,21,35,22,64,43
806,Manchester United,1729,2014/2015,10,8,20,41,15,21,22,62,37
807,Manchester United,1729,2015/2016,9,10,19,27,9,22,26,49,35


In [32]:
home_record = team_match_home.groupby(["Club","League","Season"])["result"].value_counts().unstack().fillna(0).astype(int).reset_index()


In [33]:
home_record = home_record.rename(columns = {
    "D":"D_home",
    "W":"W_home",
    "L":"L_home"
})

In [34]:
away_record = team_match_away.groupby(["Club","League","Season"])["result"].value_counts().unstack().fillna(0).astype(int).reset_index()


In [35]:
away_record = away_record.rename(columns = {
    "D":"D_away",
    "W":"W_away",
    "L":"L_away"
})

In [36]:
team_stats = team_stats.merge(
    home_record, 
    on = ["Club","League","Season"], 
    how = "left")

In [37]:
team_stats.head()

,Club,League,Season,D,L,W,GF_home,GA_home,GF_away,GA_away,GF,GA,D_home,L_home,W_home
0,1. FC Kaiserslautern,7809,2010/2011,7,14,13,25,19,23,32,48,51,6,5,6
1,1. FC Kaiserslautern,7809,2011/2012,11,19,4,12,28,12,26,24,54,5,10,2
2,1. FC Köln,7809,2008/2009,6,17,11,14,25,21,25,35,50,5,8,4
3,1. FC Köln,7809,2009/2010,11,14,9,18,29,15,13,33,42,6,8,3
4,1. FC Köln,7809,2010/2011,5,16,13,30,21,17,41,47,62,2,4,11


In [38]:
team_stats = team_stats.merge(
    away_record, 
    on = ["Club","League","Season"], 
    how = "left")

In [39]:
team_stats.head()

,Club,League,Season,D,L,W,GF_home,GA_home,GF_away,GA_away,GF,GA,D_home,L_home,W_home,D_away,L_away,W_away
0,1. FC Kaiserslautern,7809,2010/2011,7,14,13,25,19,23,32,48,51,6,5,6,1,9,7
1,1. FC Kaiserslautern,7809,2011/2012,11,19,4,12,28,12,26,24,54,5,10,2,6,9,2
2,1. FC Köln,7809,2008/2009,6,17,11,14,25,21,25,35,50,5,8,4,1,9,7
3,1. FC Köln,7809,2009/2010,11,14,9,18,29,15,13,33,42,6,8,3,5,6,6
4,1. FC Köln,7809,2010/2011,5,16,13,30,21,17,41,47,62,2,4,11,3,12,2


In [40]:
team_stats[team_stats["Club"] == "Manchester United"]

,Club,League,Season,D,L,W,GF_home,GA_home,GF_away,GA_away,GF,GA,D_home,L_home,W_home,D_away,L_away,W_away
800,Manchester United,1729,2008/2009,6,4,28,43,13,25,11,68,24,2,1,16,4,3,12
801,Manchester United,1729,2009/2010,4,7,27,52,12,34,16,86,28,1,2,16,3,5,11
802,Manchester United,1729,2010/2011,11,4,23,49,12,29,25,78,37,1,0,18,10,4,5
803,Manchester United,1729,2011/2012,5,5,28,52,19,37,14,89,33,2,2,15,3,3,13
804,Manchester United,1729,2012/2013,5,5,28,45,19,41,24,86,43,0,3,16,5,2,12
805,Manchester United,1729,2013/2014,7,12,19,29,21,35,22,64,43,3,7,9,4,5,10
806,Manchester United,1729,2014/2015,10,8,20,41,15,21,22,62,37,2,3,14,8,5,6
807,Manchester United,1729,2015/2016,9,10,19,27,9,22,26,49,35,5,2,12,4,8,7
